# Демонстрация версионирования datasets scikit-learn

В ноутбуке один и тот же набор файлов последовательно версионируется с помощью DVC и ClearML Dataset. Для демонстрации используются `load_digits` и `load_diabetes`.

Самостоятельная работа использует другие datasets и более сложный сценарий слияния, поэтому этот notebook не является готовым решением домашнего задания.

## Что будет показано

1. Детерминированное сохранение таблиц из scikit-learn.
2. Создание двух версий данных в DVC.
3. DVC remote, переключение Git-веток и `dvc checkout`.
4. Восстановление данных в чистом Git clone.
5. Создание базовой и дочерней версии ClearML Dataset.
6. Проверка Dataset IDs, parent lineage, состава и SHA-256.

Локальная часть DVC создаёт новые демонстрационные директории и не удаляет существующие проекты. ClearML-часть по умолчанию выключена, чтобы notebook не создавал объекты на сервере без явного решения пользователя.

## Подготовка окружения

Для полного выполнения нужны Python 3.12, Git, DVC, ClearML, pandas и scikit-learn. Установите зависимости в отдельное виртуальное окружение и зафиксируйте их версии.

Перед ClearML-частью настройте подключение командой `clearml-init`. Не вставляйте credentials в notebook.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as metadata
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.datasets import load_diabetes, load_digits

for package_name in ["pandas", "scikit-learn", "dvc", "clearml"]:
    try:
        print(f"{package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"{package_name}: не установлен")

## Вспомогательные функции

`run` выполняет внешнюю команду в явно указанной директории и останавливает выполнение при ошибке. `sha256_file` вычисляет хэш файла, а `describe_csv` проверяет фактическую структуру сохранённой таблицы.

In [ ]:
def run(command: list[str | Path], cwd: Path) -> subprocess.CompletedProcess[str]:
    command_as_text = [str(part) for part in command]
    print("$", " ".join(command_as_text))
    result = subprocess.run(
        command_as_text,
        cwd=cwd,
        check=True,
        text=True,
        capture_output=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    return result


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_frame(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = frame.sort_values(list(frame.columns), kind="mergesort").reset_index(drop=True)
    ordered.to_csv(path, index=False, lineterminator="\n", float_format="%.10g")


def describe_csv(path: Path) -> dict[str, object]:
    frame = pd.read_csv(path)
    return {
        "file": path.name,
        "sha256": sha256_file(path),
        "rows": len(frame),
        "columns": len(frame.columns),
        "missing": int(frame.isna().sum().sum()),
    }

## Создание исходных таблиц

В демонстрации первая версия содержит только Digits. Во второй версии добавляется Diabetes, а Digits остаётся неизменным. Время запуска входит только в имя рабочей директории и не записывается в CSV.

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
demo_root = Path.cwd() / f"sklearn_versioning_demo_{run_id}"
source_dir = demo_root / "source_tables"
dvc_repo = demo_root / "dvc_repository"
dvc_remote = demo_root / "dvc_remote"
dvc_clone = demo_root / "dvc_clean_clone"

digits_frame = load_digits(as_frame=True).frame
diabetes_frame = load_diabetes(as_frame=True).frame

digits_source = source_dir / "digits.csv"
diabetes_source = source_dir / "diabetes.csv"
save_frame(digits_frame, digits_source)
save_frame(diabetes_frame, diabetes_source)

pd.DataFrame([describe_csv(digits_source), describe_csv(diabetes_source)])

Ожидаемый результат: `digits.csv` содержит 1797 строк, `diabetes.csv` содержит 442 строки, пропущенных значений нет. SHA-256 зависит от зафиксированных версий pandas и scikit-learn, поэтому в работе нужно сохранять и хэш, и версии окружения.

# Часть 1. DVC

## Проверка инструментов

Git и DVC вызываются как внешние команды. Если команда не найдена, установите инструмент в текущее окружение и повторите ячейку.

In [ ]:
git_executable = shutil.which("git")
dvc_executable = shutil.which("dvc")

if git_executable is None:
    raise RuntimeError("Git не найден в PATH")
if dvc_executable is None:
    raise RuntimeError("DVC не найден в PATH текущего окружения")

print("Git:", git_executable)
print("DVC:", dvc_executable)

## Базовая версия DVC

Создаётся отдельный Git-репозиторий, DVC инициализируется внутри него, а `data` начинает отслеживаться как единая директория. Локальные имя и email Git задаются только для демонстрационного репозитория.

In [ ]:
dvc_repo.mkdir(parents=True)
run([git_executable, "init", "-b", "main"], dvc_repo)
run([git_executable, "config", "user.name", "MIIT Demo"], dvc_repo)
run([git_executable, "config", "user.email", "miit-demo@example.invalid"], dvc_repo)
run([dvc_executable, "init"], dvc_repo)

data_dir = dvc_repo / "data"
data_dir.mkdir()
shutil.copy2(digits_source, data_dir / "digits.csv")

run([dvc_executable, "add", "data"], dvc_repo)
run([git_executable, "add", ".dvc", ".dvcignore", ".gitignore", "data.dvc"], dvc_repo)
run([git_executable, "commit", "-m", "Add digits dataset v1"], dvc_repo)

base_commit = run([git_executable, "rev-parse", "HEAD"], dvc_repo).stdout.strip()
base_digits_hash = sha256_file(data_dir / "digits.csv")
print("Base commit:", base_commit)
print("Digits SHA-256:", base_digits_hash)

Что произошло:

- CSV сохранён в DVC cache.
- `data.dvc` хранит метаданные директории.
- `data` добавлена в `.gitignore`.
- Git-коммит хранит код управления версией, но не CSV.

In [ ]:
print((dvc_repo / "data.dvc").read_text(encoding="utf-8"))
run([git_executable, "status", "--short"], dvc_repo)
run([dvc_executable, "status"], dvc_repo)

## Локальный DVC remote

Remote создаётся рядом с репозиторием, но не внутри него. В реальном проекте это может быть S3, сетевой диск или другое общее хранилище.

In [ ]:
dvc_remote.mkdir()
run([dvc_executable, "remote", "add", "-d", "demo_storage", str(dvc_remote.resolve())], dvc_repo)
run([git_executable, "add", ".dvc/config"], dvc_repo)
run([git_executable, "commit", "-m", "Configure local DVC remote"], dvc_repo)
run([dvc_executable, "push"], dvc_repo)

Ожидаемый результат: объекты базовой версии появились в remote. `git push` и `dvc push` решают разные задачи: первый отправляет метаданные и код, второй отправляет содержимое данных.

## Вторая версия в отдельной ветке

Во второй версии к неизменному Digits добавляется Diabetes. После повторного `dvc add` изменяется `data.dvc`, а новые объекты отправляются в remote.

In [ ]:
run([git_executable, "switch", "-c", "feature/add-diabetes"], dvc_repo)
shutil.copy2(diabetes_source, data_dir / "diabetes.csv")
run([dvc_executable, "add", "data"], dvc_repo)
run([git_executable, "add", "data.dvc"], dvc_repo)
run([git_executable, "commit", "-m", "Add diabetes dataset v2"], dvc_repo)
run([dvc_executable, "push"], dvc_repo)

second_commit = run([git_executable, "rev-parse", "HEAD"], dvc_repo).stdout.strip()
second_digits_hash = sha256_file(data_dir / "digits.csv")
assert second_digits_hash == base_digits_hash
pd.DataFrame([describe_csv(path) for path in sorted(data_dir.glob("*.csv"))])

## Переключение версий

Переключение Git-ветки изменяет `data.dvc`. Команда `dvc checkout` после этого приводит workspace в соответствие с метаданными выбранной ветки.

In [ ]:
run([git_executable, "switch", "main"], dvc_repo)
run([dvc_executable, "checkout"], dvc_repo)
assert (data_dir / "digits.csv").exists()
assert not (data_dir / "diabetes.csv").exists()
assert sha256_file(data_dir / "digits.csv") == base_digits_hash
print("main содержит:", sorted(path.name for path in data_dir.iterdir()))

run([git_executable, "switch", "feature/add-diabetes"], dvc_repo)
run([dvc_executable, "checkout"], dvc_repo)
assert (data_dir / "digits.csv").exists()
assert (data_dir / "diabetes.csv").exists()
print("feature/add-diabetes содержит:", sorted(path.name for path in data_dir.iterdir()))

## Восстановление из чистого clone

Демонстрационная ветка объединяется с `main`, после чего создаётся новый clone без копирования старого workspace и cache. Данные восстанавливаются из DVC remote.

In [ ]:
run([git_executable, "switch", "main"], dvc_repo)
run([git_executable, "merge", "--ff-only", "feature/add-diabetes"], dvc_repo)
run([dvc_executable, "checkout"], dvc_repo)
run([dvc_executable, "push"], dvc_repo)

run([git_executable, "clone", str(dvc_repo.resolve()), str(dvc_clone.resolve())], demo_root)
run([dvc_executable, "pull"], dvc_clone)

clone_data = dvc_clone / "data"
clone_summary = pd.DataFrame([describe_csv(path) for path in sorted(clone_data.glob("*.csv"))])
assert set(clone_summary["file"]) == {"digits.csv", "diabetes.csv"}
clone_summary

Ожидаемый результат: новый clone содержит оба CSV, хотя Git не хранит их содержимое. Восстановление подтверждает, что данные действительно присутствуют в remote, а не только в старом локальном cache.

# Часть 2. ClearML Dataset

ClearML-часть создаёт объекты на настроенном server, поэтому она выключена по умолчанию. Укажите уникальный идентификатор и измените `run_clearml` на `True`.

Не используйте реальные credentials в значениях ячеек.

In [ ]:
run_clearml = False
student_id = "replace-with-your-id"

if run_clearml and student_id == "replace-with-your-id":
    raise ValueError("Укажите уникальный student_id перед запуском ClearML-части")

print("ClearML execution enabled:", run_clearml)

## Базовая версия ClearML Dataset

Создаётся версия `1.0.0`, содержащая только Digits. Файлы сначала добавляются, затем загружаются и только после этого версия финализируется.

In [ ]:
if run_clearml:
    from clearml import Dataset

    clearml_project = f"MIIT/MLOps/practice_3/demo/{student_id}"
    clearml_name = f"sklearn-demo-{student_id}-{run_id.lower()}"

    base_dataset = Dataset.create(
        dataset_project=clearml_project,
        dataset_name=clearml_name,
        dataset_version="1.0.0",
        dataset_tags=["demo", "base", "digits"],
        description="Base version with sklearn Digits",
    )
    base_dataset.add_files(path=digits_source)
    base_dataset.upload()
    base_dataset.finalize()
    base_id = base_dataset.id

    print("BASE_ID:", base_id)
    print("Files:", base_dataset.list_files())
else:
    print("Ячейка пропущена. Настройте ClearML и установите run_clearml=True.")

Ожидаемый результат: версия имеет статус Completed и уникальный Dataset ID. После финализации она неизменяема.

## Дочерняя версия

Версия `1.1.0` наследует Digits от базы и добавляет Diabetes. Родитель задаётся через `parent_datasets`, а не определяется по последнему открытому Dataset.

In [ ]:
if run_clearml:
    child_dataset = Dataset.create(
        dataset_project=clearml_project,
        dataset_name=clearml_name,
        dataset_version="1.1.0",
        parent_datasets=[base_id],
        dataset_tags=["demo", "child", "diabetes"],
        description="Child version that adds sklearn Diabetes",
    )
    child_dataset.add_files(path=diabetes_source)
    child_dataset.upload()
    child_dataset.finalize()
    child_id = child_dataset.id

    print("CHILD_ID:", child_id)
    print("All files:", child_dataset.list_files())
    print("Added files:", child_dataset.list_added_files())
    print("Lineage:", child_dataset.get_dependency_graph())
else:
    print("Ячейка пропущена.")

Ожидаемый результат: полный состав child содержит два CSV, но список added files содержит только Diabetes. Dependency graph показывает базовый Dataset родителем child.

## Скачивание по Dataset ID и проверка

Каждая версия получается по точному ID и скачивается в отдельную writable директорию. Это позволяет проверить состав и хэши без зависимости от поиска по имени.

In [ ]:
if run_clearml:
    clearml_downloads = demo_root / "clearml_downloads"
    base_download = clearml_downloads / "base_1_0_0"
    child_download = clearml_downloads / "child_1_1_0"

    Dataset.get(dataset_id=base_id).get_mutable_local_copy(
        target_folder=base_download, overwrite=False
    )
    Dataset.get(dataset_id=child_id).get_mutable_local_copy(
        target_folder=child_download, overwrite=False
    )

    clearml_rows = []
    for version_name, directory in [("1.0.0", base_download), ("1.1.0", child_download)]:
        for csv_path in sorted(directory.rglob("*.csv")):
            row = describe_csv(csv_path)
            row["version"] = version_name
            clearml_rows.append(row)

    clearml_summary = pd.DataFrame(clearml_rows)
    assert set(clearml_summary.query("version == '1.0.0'")["file"]) == {"digits.csv"}
    assert set(clearml_summary.query("version == '1.1.0'")["file"]) == {"digits.csv", "diabetes.csv"}
    display(clearml_summary)
else:
    print("Ячейка пропущена.")

# Сравнение наблюдений

| Наблюдение | DVC | ClearML Dataset |
|---|---|---|
| Версия связана с | Git commit и `data.dvc` | Dataset ID и lineage |
| Общее хранилище | DVC remote | ClearML File Server или настроенный storage |
| Переключение | Git branch и `dvc checkout` | Получение конкретного Dataset ID |
| Дочерняя версия | Новый Git commit или branch | Новый Dataset с `parent_datasets` |
| Проверка содержимого | `.dvc`, workspace и SHA-256 | file lists, local copy и SHA-256 |

Успешное выполнение notebook не удаляет созданные ClearML objects. Сохраните Dataset IDs, если они нужны для проверки, а ненужные демонстрационные versions удаляйте только осознанно через UI или SDK.

## Контрольные вопросы

1. Почему после `git switch` в DVC-репозитории нужен `dvc checkout`?
2. Что потеряется, если выполнить `git push`, но не выполнить `dvc push`?
3. Почему финализированный ClearML Dataset нельзя изменять?
4. Чем Dataset ID надёжнее поиска только по имени?
5. Почему для проверки CSV сохраняются и SHA-256, и форма таблицы?